# Petal on a free GPU

One run of this notebook is one round of learning for Petal, on a free GPU.
It downloads her code and memory from your GitHub repo, trains her, and saves her
back to the `petal-data` branch. Her website picks her up within about 20 minutes.

**Before the first run:** turn **Accelerator = GPU** and **Internet = On** in the notebook
settings, and add a Secret named `GITHUB_TOKEN` (Add-ons > Secrets). Full steps are in PETAL_GPU.md.

For long runs use **Save Version > Save & Run All (Commit)**, so it keeps running on
Kaggle's servers even if you close the app.

In [ ]:
import os, sys, base64, subprocess

# ---- 1. YOUR SETTINGS ------------------------------------------------------
REPO       = "benjaminkazibwe95-code/daisy"   # your GitHub repo
HARVEST_MIN = "20"    # minutes spent reading new pages first
TRAIN_MIN   = "150"   # minutes of GPU training (first test: try "15")
BATCH       = "128"   # examples per step; a GPU can take much more than the 32 used on GitHub

# ---- 2. THE GITHUB TOKEN (kept in Secrets, never typed in here) -----------
token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        pass
if not token:
    raise SystemExit("Add a Secret named GITHUB_TOKEN (Add-ons > Secrets), then run again.")

os.environ.update(PETAL_REPO=REPO, GITHUB_TOKEN=token, PETAL_HARVEST_MIN=HARVEST_MIN,
                  PETAL_TRAIN_MIN=TRAIN_MIN, PETAL_BATCH=BATCH)

# ---- 3. SHOW THE GPU, FETCH THE RUNNER FROM YOUR REPO, START IT -----------
subprocess.run(["nvidia-smi", "-L"])
header = "AUTHORIZATION: basic " + base64.b64encode(f"x-access-token:{token}".encode()).decode()
boot = "/tmp/petal_boot"
subprocess.run(["rm", "-rf", boot])
r = subprocess.run(["git", "-c", f"http.extraheader={header}", "clone", "--depth", "1",
                    f"https://github.com/{REPO}.git", boot], capture_output=True, text=True)
if r.returncode != 0:
    raise SystemExit("Could not download your repo. Check the repo name and that the token can read it.")
code = subprocess.run([sys.executable, f"{boot}/petal_gpu.py"]).returncode
if code != 0:
    raise SystemExit(f"Petal's round stopped (code {code}). Read the messages above.")
print("Petal's round finished.")
